# Fine-tuning RF-DETR-S sur American Sign Language Letters

**Plateforme :** Google Colab (Runtime > Change runtime type > T4 GPU)

**Objectif :** Entraîner un modèle RF-DETR-S sur le dataset ASL en parallèle de YOLO pour la comparaison.

**Durée estimée :** ~2-3h sur T4 (30 epochs, batch_size=4, grad_accum_steps=4).

## 1. Vérification GPU

In [1]:
!nvidia-smi

Fri May  8 20:35:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Installation des dépendances

In [6]:
!pip install -q rfdetr roboflow supervision
!pip install -q "rfdetr[train,loggers]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Téléchargement du dataset ASL au format COCO

In [3]:
import os
from roboflow import Roboflow

API_KEY = os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("david-lee-d0rhs").project("american-sign-language-letters")
version = project.version(6)
dataset = version.download("coco")

print("Dataset téléchargé dans:", dataset.location)
!ls {dataset.location}

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to American-Sign-Language-Letters-6 in coco:: 100%|██████████| 728/728 [00:00<00:00, 1897.44it/s]


Dataset téléchargé dans: /content/American-Sign-Language-Letters-6
README.dataset.txt  README.roboflow.txt  test  train  valid


In [4]:
# RF-DETR attend les sous-dossiers train/, valid/, test/ avec _annotations.coco.json
# Roboflow exporte exactement ce format avec download("coco")
import json
from pathlib import Path

train_ann = Path(dataset.location) / "train" / "_annotations.coco.json"
with open(train_ann) as f:
    coco = json.load(f)
print(f"Nombre de classes: {len(coco['categories'])}")
print(f"Classes: {[c['name'] for c in coco['categories']]}")
print(f"Nombre d'images train: {len(coco['images'])}")
print(f"Nombre d'annotations train: {len(coco['annotations'])}")

Nombre de classes: 27
Classes: ['Letters', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Nombre d'images train: 504
Nombre d'annotations train: 504


## 4. Entraînement RF-DETR-S

- 30 epochs (RF-DETR converge plus vite que YOLO sur petit dataset)
- batch_size=4, grad_accum_steps=4 (effective batch=16, T4-friendly)
- Sauvegarde tous les 5 epochs

In [8]:
from rfdetr import RFDETRSmall
import torch

torch.manual_seed(42)

model = RFDETRSmall()

model.train(
    dataset_dir=dataset.location,
    epochs=30,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    output_dir='./output_rfdetr_asl',
    early_stopping=True,
    early_stopping_patience=10,
    seed=42,
)

[2026-05-08 20:57:39] [INFO] rf-detr - File rf-detr-small.pth already exists with correct MD5 hash.


[2026-05-08 20:57:39] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-08 20:57:39] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-05-08 20:57:40] [INFO] rf-detr - File rf-detr-small.pth already exists with correct MD5 hash.


[2026-05-08 20:57:42] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-08 20:57:42] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-05-08 20:57:43] [INFO] rf-detr - File rf-detr-small.pth already exists with correct MD5 hash.


...

[2026-05-08 22:04:17] [INFO] rf-detr - Best EMA mAP improved to 0.9348 (epoch 26)
[2026-05-08 22:06:40] [INFO] rf-detr - Best regular mAP saved to /content/output_rfdetr_asl/checkpoint_best_regular.pth (epoch 27)
[2026-05-08 22:09:19] [INFO] rf-detr - Best EMA mAP improved to 0.9349 (epoch 28)


INFO:pytorch_lightning.callbacks.early_stopping:Metric __rfdetr_effective_map__ improved by 0.002 >= min_delta = 0.001. New best score: 0.937


[2026-05-08 22:11:50] [INFO] rf-detr - Best EMA mAP improved to 0.9366 (epoch 29)


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


[2026-05-08 22:12:09] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.9160, ema=0.9366)


## 5. Évaluation sur le test set

In [9]:
from rfdetr import RFDETRSmall
import os, glob

# Recherche du best checkpoint (recursive si besoin)
candidates = sorted(glob.glob('**/checkpoint_best_total.pth', recursive=True),
                    key=os.path.getmtime, reverse=True)
if not candidates:
    candidates = sorted(glob.glob('**/checkpoint*.pth', recursive=True),
                        key=os.path.getmtime, reverse=True)

assert candidates, "Aucun checkpoint trouvé — le training a-t-il bien terminé ?"
best_ckpt = candidates[0]
print(f"Utilisation de: {best_ckpt}")

best_model = RFDETRSmall(pretrain_weights=best_ckpt)

# Évaluation - l'API peut varier selon la version de rfdetr.
# Si .evaluate() ne marche pas, on fera l'éval dans le notebook 04 via supervision.
try:
    from pathlib import Path
    metrics = best_model.evaluate(dataset_dir=dataset.location, split='test')
    print('Métriques:', metrics)
except Exception as e:
    print(f"[INFO] .evaluate() non disponible ou a échoué ({e}).")
    print("L'évaluation comparative sera réalisée dans le notebook 04 via supervision.")

[2026-05-08 22:12:39] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-08 22:12:39] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


Utilisation de: output_rfdetr_asl/checkpoint_best_total.pth


[2026-05-08 22:12:39] [WARNING] rf-detr - Checkpoint has 27 classes but model is configured for 90. Using checkpoint class count (27). Pass num_classes=27 to suppress this warning.


[INFO] .evaluate() non disponible ou a échoué ('RFDETRSmall' object has no attribute 'evaluate').
L'évaluation comparative sera réalisée dans le notebook 04 via supervision.


## 6. Temps d'inférence

In [10]:
import time
import glob
from PIL import Image

test_images = sorted(glob.glob(f"{dataset.location}/test/*.jpg"))[:50]
print(f"Mesure sur {len(test_images)} images")

img0 = Image.open(test_images[0])
_ = best_model.predict(img0)  # warmup

start = time.time()
for img_path in test_images:
    img = Image.open(img_path)
    _ = best_model.predict(img)
elapsed = time.time() - start
print(f"Temps moyen par image: {1000*elapsed/len(test_images):.2f} ms")
print(f"FPS estimé: {len(test_images)/elapsed:.1f}")

[2026-05-08 22:12:44] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


Mesure sur 50 images
Temps moyen par image: 141.20 ms
FPS estimé: 7.1


## 7. Sauvegarde et téléchargement

In [11]:
import shutil, os
from google.colab import files

size_mb = os.path.getsize(best_ckpt) / 1024 / 1024
print(f"Taille modèle best: {size_mb:.2f} MB")

# Archive du dossier de sortie (où qu'il soit)
output_dir = os.path.dirname(best_ckpt)
shutil.make_archive('rfdetr_asl_run', 'zip', output_dir)
print(f"Archive créée: rfdetr_asl_run.zip (depuis {output_dir})")

files.download(best_ckpt)
files.download('rfdetr_asl_run.zip')

Taille modèle best: 121.82 MB
Archive créée: rfdetr_asl_run.zip (depuis output_rfdetr_asl)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>